# 01 — Setup & Verification
Install dependencies and verify AWS Bedrock access for Claude claude-opus-4-6.

In [ ]:
%pip install pymupdf boto3 Pillow reportlab python-dotenv --prefer-binary -q

In [ ]:
from utils import get_logger, extract_json
logger = get_logger("01_setup")

In [ ]:
# Verify imports
import fitz
import boto3
from PIL import Image
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph
import json, os, sys, base64

# Add project root to path so we can import from models/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

logger.info("PyMuPDF  : %s", fitz.__version__)
logger.info("boto3    : %s", boto3.__version__)
logger.info("All imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────
from models import get_bedrock_client, resolve_model_id, get_available_models

# Choose model: change this to any key from config/models.json
# e.g. "sonnet 3.7", "haiku 4.5", "sonnet 4.5", "opus 4.6"
BEDROCK_MODEL = resolve_model_id()  # default from config/models.json

BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), ".."))
INPUT_FOLDER  = os.path.join(BASE_DIR, "input_folder")
OUTPUT_FOLDER = os.path.join(BASE_DIR, "output_folder")
TEMP_FOLDER   = os.path.join(BASE_DIR, "temp_images")
PAGE_DPI      = 150

logger.info("Available models: %s", list(get_available_models().keys()))
logger.info("Input    : %s", INPUT_FOLDER)
logger.info("Output   : %s", OUTPUT_FOLDER)
logger.info("Temp     : %s", TEMP_FOLDER)
logger.info("Model    : %s", BEDROCK_MODEL)

In [ ]:
# Verify Bedrock connectivity
client = get_bedrock_client()

logger.info("Testing Bedrock connection...")
test_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 64,
    "messages": [{"role": "user", "content": "Reply with: Bedrock connection OK"}]
}

response = client.invoke_model(
    modelId=BEDROCK_MODEL,
    body=json.dumps(test_body),
    contentType="application/json",
    accept="application/json",
)
result = json.loads(response["body"].read())
logger.info("Bedrock response: %s", result["content"][0]["text"])

In [ ]:
# ── Smoke test — dummy replacement on a synthetic image ────────
from PIL import Image, ImageDraw
import io

logger.info("Building synthetic test image with fake PII...")
img = Image.new("RGB", (600, 220), color="white")
draw = ImageDraw.Draw(img)
draw.text((10, 10),  "Patient: John Smith",           fill="black")
draw.text((10, 40),  "DOB: 01/15/1985",               fill="black")
draw.text((10, 70),  "Diagnosis: Type 2 Diabetes",    fill="black")
draw.text((10, 100), "SSN: 123-45-6789",              fill="black")
draw.text((10, 130), "Email: john.smith@email.com",   fill="black")
draw.text((10, 160), "Notes: Follow-up in 3 months",  fill="black")
draw.text((10, 190), "Referring Dr: John Smith",      fill="black")

buf = io.BytesIO()
img.save(buf, format="PNG")
img_b64 = base64.standard_b64encode(buf.getvalue()).decode()
logger.debug("Test image encoded (%d bytes base64)", len(img_b64))

PROMPT = """You are a data-sanitization assistant. Your task is to take the provided document \
and produce a new, clean version in which all PII (Personally Identifiable Information) and \
PHI (Protected Health Information) are fully redacted and replaced with realistic but \
completely fictitious dummy records.

Identify and replace:
- Full names (first, last, middle, initials)
- Email addresses
- Phone numbers
- Social Security Numbers (SSN) or national identifiers
- Physical mailing addresses
- Dates of Birth (DOB)
- Medical record numbers or identifiers
- Specific medical diagnoses or conditions tied to individuals

Requirements:
1. Replace each sensitive item with a consistent, fictitious dummy value.
   If the same name appears multiple times, use the SAME replacement throughout.
2. Dummy values must follow valid formats but must not correspond to real individuals.
3. Maintain the original meaning, readability, and structure — only sensitive data is substituted.

Return your response as valid JSON with exactly this structure (no markdown fences):
{
  "sanitized_text": "<full sanitized page text, preserving line breaks as \\n>",
  "mapping": [
    {"original_masked": "J*** S***", "replacement": "Alex Carter", "type": "Name"},
    ...
  ]
}

Now process the following document:"""

body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 2048,
    "messages": [{
        "role": "user",
        "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
            {"type": "text",  "text": PROMPT}
        ]
    }]
}

logger.info("Sending test image to Bedrock...")
resp = client.invoke_model(
    modelId=BEDROCK_MODEL, body=json.dumps(body),
    contentType="application/json", accept="application/json"
)
raw = json.loads(resp["body"].read())["content"][0]["text"]
logger.debug("Raw model response (first 300 chars): %s", raw[:300].replace("\n", "\\n"))

output = extract_json(raw)

logger.info("Smoke test complete — %d mapping entries returned", len(output["mapping"]))
print("\n=== Sanitized text ===")
print(output["sanitized_text"])
print("\n=== Mapping table ===")
print(f"{'Original (masked)':<25} {'Replacement':<25} {'Type'}")
print("-" * 65)
for row in output["mapping"]:
    print(f"{row['original_masked']:<25} {row['replacement']:<25} {row['type']}")